# 🦜🔗 LangChain — Pilar 2: Data Connection / RAG

> **Versões:** `langchain-core==1.4.0` | `langchain-openai==1.1.10` | `langchain-community==1.0.0` | `faiss-cpu` | Maio/2026

---

## O problema que o RAG resolve

Um LLM foi treinado com dados até uma certa data histórica. Ele **não possui conhecimento nativo** sobre:
- Documentos internos da sua empresa
- Informações de fatos muito recentes
- Dados privados e confidenciais

**RAG (Retrieval-Augmented Generation)** resolve isso: antes de submeter a pergunta ao modelo, o sistema *busca* os trechos de documentos relevantes e os *injeta* no contexto do prompt.

### O pipeline RAG completo:

```
PREPARAÇÃO (feita uma vez):
Documentos → Loader → Splitter → Embeddings → VectorStore

CONSULTA (a cada pergunta):
Pergunta → Embeddings → Retrieval → Prompt + Contexto → LLM → Resposta
```

---

## ⚙️ Instalação

In [ ]:
%pip install -qU langchain-core==1.4.0 langchain-openai==1.1.10 langchain-community==1.0.0 langchain-text-splitters faiss-cpu numpy tiktoken python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv('OPENAI_API_KEY'):
    raise EnvironmentError(
        'OPENAI_API_KEY não encontrada.\n'
        'Crie um arquivo .env na raiz do projeto com:\n'
        'OPENAI_API_KEY=sk-...'
    )

print('Ambiente configurado!')

---
## 1. 📄 Document Loaders — Carregando dados brutos

Document Loaders transformam diversas fontes de dados em **objetos `Document`** padronizados.

Cada `Document` é composto por:
- `page_content`: o texto extraído da fonte
- `metadata`: um dicionário com metadados de origem (nome do arquivo, número da página, URL, etc.)

### Fontes disponíveis:

| Loader | Fonte | Pacote |
|---|---|---|
| `TextLoader` | Arquivo `.txt` | langchain-community |
| `PyPDFLoader` | Arquivo `.pdf` | langchain-community |
| `WebBaseLoader` | Página web | langchain-community |
| `CSVLoader` | Arquivo `.csv` | langchain-community |
| `DirectoryLoader` | Pasta inteira | langchain-community |

In [ ]:
# Vamos criar um arquivo de texto de exemplo para usar no tutorial
conteudo_exemplo = """LangChain: O Framework para Aplicações com LLMs

LangChain é um framework open-source criado em 2022 por Harrison Chase.
Ele foi criado com o objetivo de facilitar o desenvolvimento de aplicações
que usam Large Language Models (LLMs).

Principais características do LangChain:
O LangChain oferece abstrações para models, prompts, chains, memory, agents e tools.
A versão mais recente usa LCEL (LangChain Expression Language) com o operador pipe.
O framework suporta mais de 50 provedores de LLMs diferentes.

Casos de uso comuns:
RAG (Retrieval-Augmented Generation) é o caso de uso mais popular do LangChain.
Chatbots com memória de conversação são amplamente usados em produção.
Agentes autônomos que usam ferramentas externas estão crescendo rapidamente.

Ecossistema LangChain:
LangChain Core fornece as abstrações base do framework.
LangGraph é usado para fluxos complexos com estado e ciclos.
LangSmith oferece observabilidade e rastreamento de chamadas ao LLM.
"""

with open('documento_exemplo.txt', 'w', encoding='utf-8') as f:
    f.write(conteudo_exemplo)

print('Arquivo criado: documento_exemplo.txt')

In [ ]:
from langchain_community.document_loaders import TextLoader

# Carrega o arquivo de texto
loader = TextLoader('documento_exemplo.txt', encoding='utf-8')
documentos = loader.load()

print(f'Número de documentos carregados: {len(documentos)}')
print(f'Tipo: {type(documentos[0])}')
print(f'Metadados: {documentos[0].metadata}')
print(f'\nPrimeiros 200 caracteres do conteúdo:')
print(documentos[0].page_content[:200])

In [ ]:
# Exemplo com múltiplos documentos criados diretamente
from langchain_core.documents import Document

docs_manuais = [
    Document(
        page_content='Python foi criado por Guido van Rossum e lançado em 1991.',
        metadata={'fonte': 'wikipedia', 'topico': 'python'}
    ),
    Document(
        page_content='Python é uma das linguagens mais populares para Data Science e Machine Learning.',
        metadata={'fonte': 'survey_2025', 'topico': 'python'}
    ),
    Document(
        page_content='Pandas é a biblioteca mais usada para manipulação de dados em Python.',
        metadata={'fonte': 'docs_pandas', 'topico': 'pandas'}
    )
]

print(f'Total de documentos: {len(docs_manuais)}')
for i, doc in enumerate(docs_manuais):
    print(f'  Doc {i+1}: {doc.page_content[:60]}... | meta: {doc.metadata}')

---
## 2. ✂️ Text Splitters — Quebrando em pedaços

Modelos de linguagem têm um limite rígido de tokens por requisição. Documentos longos precisam ser fracionados em pedaços menores (**chunks**) antes de serem indexados.

### Parâmetros fundamentais:

| Parâmetro | Significado | Valor prático |
|---|---|---|
| `chunk_size` | Tamanho máximo de cada chunk | 500 a 2000 caracteres (ou tokens) |
| `chunk_overlap` | Sobreposição entre chunks adjacentes — preserva contexto semântico nas bordas | 10% a 20% do `chunk_size` |

### Qual splitter usar?

| Splitter | Critério de divisão | Quando usar |
|---|---|---|
| `CharacterTextSplitter` | Um único separador configurável | Textos com estrutura previsível (ex: parágrafos duplos) |
| `RecursiveCharacterTextSplitter` | Hierarquia de separadores com fallback automático | **Caso geral — recomendado na maioria das situações** |
| `TokenTextSplitter` | Tokens reais do tokenizador do modelo | Controle preciso do limite de contexto em produção |

### 2.1 CharacterTextSplitter — Separador único

O splitter mais direto: divide o texto usando **exatamente um separador** configurável. Se o separador não aparecer com frequência suficiente, os chunks podem ultrapassar o `chunk_size`. Use quando o formato do texto é previsível.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter_char = CharacterTextSplitter(
    separator='\n\n',      # divide apenas em parágrafos duplos
    chunk_size=300,
    chunk_overlap=50,
    length_function=len
)

chunks_char = splitter_char.split_documents(documentos)

print(f'CharacterTextSplitter — {len(chunks_char)} chunk(s) gerado(s)\n')
for i, chunk in enumerate(chunks_char):
    print(f'  Chunk {i+1} ({len(chunk.page_content)} chars): {chunk.page_content[:80]}...')

### 2.3 TokenTextSplitter — Divisão por tokens reais

Em produção, o limite do modelo é em **tokens**, não em caracteres. Um token equivale aproximadamente a 4 caracteres em inglês, mas varia por idioma e conteúdo. O `TokenTextSplitter` usa o tokenizador `tiktoken` da OpenAI para garantir que nenhum chunk exceda o limite real do modelo.

> Use este splitter quando o controle exato do número de tokens for crítico — por exemplo, para evitar erros de `context_length_exceeded` em produção.

In [ ]:
from langchain_text_splitters import TokenTextSplitter

splitter_tokens = TokenTextSplitter(
    model_name='gpt-4o',  # usa o tokenizador do modelo-alvo
    chunk_size=100,        # máximo de 100 tokens por chunk
    chunk_overlap=10       # 10 tokens de sobreposição
)

chunks_tokens = splitter_tokens.split_documents(documentos)

print(f'TokenTextSplitter — {len(chunks_tokens)} chunk(s) gerado(s)\n')
for i, chunk in enumerate(chunks_tokens):
    print(f'  Chunk {i+1}: {chunk.page_content[:120]}...')

### 2.2 RecursiveCharacterTextSplitter — Separadores hierárquicos (recomendado)

Tenta dividir o texto em uma **hierarquia de separadores**, do mais estrutural ao mais granular:

```
1º: Parágrafos (\n\n)  →  2º: Linhas (\n)  →  3º: Espaços ( )  →  4º: Caracteres
```

Se o chunk gerado ainda ultrapassar o `chunk_size`, o próximo separador da lista é tentado automaticamente — garantindo que nenhum chunk exceda o limite, mesmo em textos sem estrutura clara.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tenta quebrar o texto na sequência inteligente de separadores:
# 1º: Parágrafos (\n\n) -> 2º: Linhas (\n) -> 3º: Palavras (espaço) -> 4º: Caracteres
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        # máximo de 300 caracteres por chunk
    chunk_overlap=50,      # 50 caracteres de sobreposição semântica
    length_function=len,   # função de medição padrão
    add_start_index=True   # grava a posição inicial de caractere nos metadados
)

# Divide os documentos
chunks = splitter.split_documents(documentos)

print(f'Documento original: 1 documento')
print(f'Após split: {len(chunks)} chunks')
print()

for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} (tamanho: {len(chunk.page_content)} chars) ---')
    print(chunk.page_content)
    print(f'Meta: {chunk.metadata}')
    print()

In [ ]:
# Visualizando a sobreposição entre chunks
if len(chunks) >= 2:
    print('Final do chunk 1:')
    print(repr(chunks[0].page_content[-60:]))
    print('\nInício do chunk 2:')
    print(repr(chunks[1].page_content[:60]))
    print('\n→ Note que há sobreposição de texto entre os chunks!')

---
## 3. 🔢 Embeddings — Texto → Números

Vetores de **Embeddings** representam o **significado semântico** profunda de um bloco textual convertido em uma lista matemática de números reais de alta dimensão.

```
'gato'  → [0.02, -0.15, 0.34, ...] (1536 dimensões)
'felino' → [0.03, -0.14, 0.35, ...] (vetores muito próximos no espaço matemático)
'carro' → [0.45,  0.82, -0.21, ...] (vetor distante/diferente)
```

In [ ]:
from langchain_openai import OpenAIEmbeddings

# text-embedding-3-small: veloz, otimizado e econômico (1536 dimensões)
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')

# Gera embedding para um único texto
vetor = embeddings_model.embed_query('O que é machine learning?')

print(f'Tipo do vetor: {type(vetor)}')
print(f'Dimensões: {len(vetor)}')
print(f'Primeiros 5 valores: {[round(v, 4) for v in vetor[:5]]}')

In [ ]:
import numpy as np

# Demonstração: textos similares têm vetores próximos
textos = [
    'gato',
    'felino doméstico',
    'cachorro',
    'automóvel'
]

vetores = embeddings_model.embed_documents(textos)

# Calcula similaridade cosseno entre 'gato' e os outros
def similaridade_cosseno(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

vetor_gato = vetores[0]
print('Similaridade com "gato":')
for texto, vetor in zip(textos[1:], vetores[1:]):
    sim = similaridade_cosseno(vetor_gato, vetor)
    print(f'  "{texto}": {sim:.4f}')

---
## 4. 🗄️ VectorStores — O banco de dados semântico

Um VectorStore armazena os **embeddings** calculados e permite indexar milhares de chunks para buscas indexadas.

| VectorStore | Tipo | Quando usar |
|---|---|---|
| **FAISS** | Em memória (local) | Prototipagem rápida, datasets limitados |
| **Chroma** | Local persistido em disco | Desenvolvimento, projetos locais de médio porte |
| **Pinecone** | Banco em nuvem gerenciado | Ambientes produtivos de larga escala |

Utilizaremos o **FAISS** estruturado localmente em memória física.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')

# Cria o VectorStore a partir dos chunks indexados
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings_model
)

print(f'VectorStore criado!')
print(f'Total de vetores indexados: {vectorstore.index.ntotal}')

In [ ]:
# Busca por similaridade semântica
pergunta = 'Quais são os casos de uso do LangChain?'

# Retorna os k chunks mais relevantes
resultados = vectorstore.similarity_search(pergunta, k=2)

print(f'Pergunta: "{pergunta}"')
print(f'Top {len(resultados)} chunks mais relevantes:\n')

for i, doc in enumerate(resultados):
    print(f'--- Resultado {i+1} ---')
    print(doc.page_content)
    print(f'Metadados: {doc.metadata}')
    print()

In [ ]:
# Busca com scores de similaridade
resultados_com_score = vectorstore.similarity_search_with_score(pergunta, k=3)

print('Resultados com score (menor = mais similar no FAISS):\n')
for doc, score in resultados_com_score:
    print(f'Score: {score:.4f} | {doc.page_content[:80]}...')

---
## 5. 🔍 Retrieval — Buscando o que importa

O **Retriever** unifica a interface padrão de buscas do LangChain, isolando a lógica interna do banco de vetores de forma modular.

Um Retriever implementa o método estruturado `.invoke(query: str)` e devolve de forma limpa uma estrutura do tipo `List[Document]`.

In [ ]:
# Converte o VectorStore em um Retriever padronizado
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

print(f'Tipo do retriever: {type(retriever).__name__}')

# Usando o retriever
docs_relevantes = retriever.invoke('O que é LangGraph?')

print(f'\nDocumentos encontrados: {len(docs_relevantes)}')
for doc in docs_relevantes:
    print(f'  → {doc.page_content[:100]}...')

---
## 6. 🚀 Pipeline RAG Completo com LCEL

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# 1. Prompt RAG — inclui o contexto recuperado
prompt_rag = ChatPromptTemplate.from_messages([
    ('system', """Você é um assistente que responde perguntas com base APENAS no contexto fornecido.
Se a resposta não estiver explicitamente contida no contexto, responda exatamente: 'Não encontrei essa informação nos documentos.'

Contexto:
{contexto}"""),
    ('human', '{pergunta}')
])

# 2. Função para formatar os documentos recuperados
def formatar_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# 3. LLM
llm = ChatOpenAI(model='gpt-4o', temperature=0)

# 4. Chain RAG com LCEL
chain_rag = (
    {
        'contexto': retriever | formatar_docs,  # busca e formata os docs
        'pergunta': RunnablePassthrough()        # passa a pergunta adiante sem modificações
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

print('Chain RAG montada!')
print('Tipo:', type(chain_rag).__name__)

In [ ]:
# Testando a chain RAG
perguntas = [
    'Quem criou o LangChain?',
    'O que é LangGraph e para que serve?',
    'Qual é o caso de uso mais popular do LangChain?',
    'Quantas estrelas o LangChain tem no GitHub?'  # não está nos docs
]

for pergunta in perguntas:
    print(f'❓ {pergunta}')
    resposta = chain_rag.invoke(pergunta)
    print(f'💬 {resposta}')
    print()

### ⚠️ Nota de Segurança Importante sobre Persistência Local

Ao carregar índices salvos localmente com o método `load_local()`, o parâmetro `allow_dangerous_deserialization=True` é obrigatoriamente exigido pelo LangChain. 

**Por que isso ocorre?** O arquivo gerado utiliza internamente a biblioteca de serialização nativa do Python (`pickle`). Carregar arquivos binários de origens externas, desconhecidas ou não confiáveis pode acarretar a **execução de código arbitrário malicioso** em sua máquina servidor. Só utilize este parâmetro ativo se você mesmo gerou e salvou localmente o arquivo de índices.

In [ ]:
# Salvando o VectorStore localmente
vectorstore.save_local('meu_vectorstore')
print('VectorStore salvo em: ./meu_vectorstore/')

# Carregando novamente sob os critérios de segurança revisados
vectorstore_carregado = FAISS.load_local(
    'meu_vectorstore',
    embeddings=embeddings_model,
    allow_dangerous_deserialization=True  # Ativo sob a garantia de origem confiável do arquivo local
)
print(f'VectorStore carregado! Vetores ativos: {vectorstore_carregado.index.ntotal}')

---
## 📚 Resumo do Pipeline RAG

| Etapa | Classe / Recurso | Função |
|---|---|---|
| **Carregar** | `TextLoader`, `WebBaseLoader` | Transforma origens externas de dados textuais em objetos `Document` |
| **Dividir** | `RecursiveCharacterTextSplitter` | Segmenta strings longas em chunks gerenciáveis controlando tamanho e overlap |
| **Embeddings** | `OpenAIEmbeddings` | Processa e converte strings em vetores matemáticos numéricos de alta dimensionalidade |
| **Armazenar** | `FAISS.from_documents()` | Indexa os vetores e conteúdos textuais para buscas lógicas rápidas |
| **Buscar** | `vectorstore.as_retriever()` | Converte o armazenamento em uma interface agnóstica de buscas semânticas |
| **Gerar** | Chain RAG com LCEL | Integra os dados retornados contextualizados no prompt para alimentar a resposta final da LLM |

> **Próximo passo:** `03_orchestration_composition.ipynb` — Chains Complexas, Memória Avançada, Agentes Autônomos e Tools.